In [ ]:
import os
import json
import pandas as pd
import requests

# =====================================================================
# 1. FUNÇÕES AUXILIARES COM CACHE DE IP
# =====================================================================

cache_paises = {}

def obter_pais_ip(ip: str) -> str:
    """Consulta o país de origem de um IP utilizando cache local."""
    ip_str = str(ip).strip()
    if ip_str in cache_paises:
        return cache_paises[ip_str]
    
    try:
        res = requests.get(f"http://ip-api.com/json/{ip_str}?fields=country", timeout=1.5)
        if res.status_code == 200:
            pais = res.json().get("country", "Desconhecido")
            cache_paises[ip_str] = pais
            return pais
    except Exception:
        pass
    
    cache_paises[ip_str] = "Desconhecido"
    return "Desconhecido"

# =====================================================================
# 2. CARGA E PROCESSAMENTO DOS LOGS (VETORIZADO - MUITO RÁPIDO)
# =====================================================================

file_path = 'three_months.csv'

if not os.path.exists(file_path):
    print(f"❌ Erro: Arquivo '{file_path}' não encontrado no diretório.")
else:
    df = pd.read_csv(file_path)

    # 1. Filtrar requisições ao endpoint vulnerável
    df_invoices = df[df['http_uri'].str.contains('/invoices/search', na=False, case=False)].copy()

    # 2. Extração Vetorizada (roda em milissegundos)
    df_invoices['invoice_id'] = df_invoices['http_uri'].str.extract(r'invoice_id=([^&]+)', expand=False)
    df_invoices['site_id'] = df_invoices['http_uri'].str.extract(r'site_id=([^&]+)', expand=False)
    df_invoices['authtoken'] = df_invoices['http_uri'].str.extract(r'(?:token|authtoken)=([^&]+)', expand=False)

    print(f"✅ Registros totais analisados em /invoices/search: {len(df_invoices)}\n")

    # =====================================================================
    # 3. EXTRAÇÃO DAS MÉTRICAS FORENSES
    # =====================================================================

    # 1. Top 20 IPs
    top_20_ips = df_invoices['source_ip'].value_counts().head(20)

    # 2. Top Países
    top_paises = {}
    for ip, count in top_20_ips.items():
        pais = obter_pais_ip(str(ip))
        top_paises[pais] = top_paises.get(pais, 0) + int(count)

    # 3. Top 10 AuthTokens
    top_10_tokens = df_invoices['authtoken'].dropna().value_counts().head(10)

    # 4. Total de faturas únicas
    total_invoices = df_invoices['invoice_id'].nunique()

    # 5. Sites mais afetados
    top_sites = df_invoices['site_id'].dropna().value_counts()

    # 6. Timeline do Incidente
    df_invoices['timestamp'] = pd.to_datetime(df_invoices['timestamp'], errors='coerce')
    inicio_ataque = df_invoices['timestamp'].min()
    fim_ataque = df_invoices['timestamp'].max()

    # Exibição dos Resultados
    print("=======================================================")
    print("📋 RESPOSTAS DO DESAFIO FORENSE (ANÁLISE DE DADOS)")
    print("=======================================================\n")

    print("--- 1. Top 20 IPs com mais requisições ---")
    print(top_20_ips, "\n")

    print("--- 2. Top Países de Origem ---")
    print(json.dumps(top_paises, indent=2, ensure_ascii=False), "\n")

    print("--- 3. Top 10 AuthTokens utilizados ---")
    print(top_10_tokens, "\n")

    print("--- 4. Total de invoice_id únicos / usuários afetados ---")
    print(f"Total: {total_invoices}\n")

    print("--- 5. Sites mais afetados (site_id) ---")
    print(top_sites, "\n")

    print("--- 6. Timeline do Incidente ---")
    print(f"Início: {inicio_ataque}")
    print(f"Fim:    {fim_ataque}\n")

KeyboardInterrupt: 